# Código para limpiar y transformar el precio OMIE

## Version 0

In [ ]:
import pandas as pd

# =========================
# 1. CARGA del CSV
# =========================

df_price = pd.read_csv("OMIEPrecio2020_2024.csv", sep=";")

# Eliminar columnas vacias si hay
df_price = df_price.dropna(axis=1, how="all")

# =========================
# 2. QUEDARNOS SOLO CON PRECIO
# =========================

#filtrar solo el precio de España (SP)
df_price = df_price[df_price["CONCEPT"] == "PRICE_SP"]

# =========================
# 3. PASAR DE ANCHO A LARGO
# =========================

# En lugar de tener las horas en columnas, tener una fila por hora
hour_cols = [col for col in df_price.columns if col.startswith("H")]

df_long = df_price.melt(
    id_vars=["DATE"],
    value_vars=hour_cols,
    var_name="hour",
    value_name="price"
)

# =========================
# 4. LIMPIEZA HORA
# =========================

# Extraer número de hora
df_long["hour"] = df_long["hour"].str.replace("H", "").astype(int)

# Tenemos 1-24 y lo lo pasamos a formato 0-23 
df_long["hour"] = df_long["hour"] - 1

# =========================
# 5. CREAR DATETIME
# =========================

df_long["datetime"] = pd.to_datetime(df_long["DATE"]) + \
                      pd.to_timedelta(df_long["hour"], unit="h")

# =========================
# 6. LIMPIEZA FINAL
# =========================


df_long = df_long.drop(columns=["DATE"])
df_long["price"] = pd.to_numeric(df_long["price"], errors="coerce")

df_long = df_long.sort_values("datetime").reset_index(drop=True)

# =========================
# RESULTADO FINAL
# =========================

print(df_long.head())
print(df_long.info())

# =========================
# Guardarlo en un CSV
# =========================

df_long.to_csv("limpiezaPrecio0_Omie2020_2024.csv", sep=';', index=None)


   hour  price            datetime
0     0  41.88 2020-01-01 00:00:00
1     1  38.60 2020-01-01 01:00:00
2     2  36.55 2020-01-01 02:00:00
3     3  32.32 2020-01-01 03:00:00
4     4  30.85 2020-01-01 04:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45675 entries, 0 to 45674
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   hour      45675 non-null  int64         
 1   price     43848 non-null  float64       
 2   datetime  45675 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.0 MB
None


## Version 1

Intentar eliminar la hora repetida del cambio de hora en otoño
Usamos la media entre la primera hora repetida y la segunda

In [ ]:
import pandas as pd

# =========================
# 1. CARGAR CSV
# =========================

df = pd.read_csv("OMIEPrecio2020_2024.csv", sep=";")

# =========================
# 2. FILTRAR PRECIO
# =========================

df = df[df["CONCEPT"] == "PRICE_SP"]

# =========================
# 3. PASAR A FORMATO LARGO
# =========================

hour_cols = [col for col in df.columns if col.startswith("H")]

df_long = df.melt(
    id_vars=["DATE"],
    value_vars=hour_cols,
    var_name="hour",
    value_name="price"
)

# =========================
# 4. LIMPIAR HORA
# =========================

# Quitar la H
df_long["hour"] = df_long["hour"].str.replace("H", "")

# Convertir a numero
df_long["hour"] = pd.to_numeric(df_long["hour"], errors="coerce")

# Eliminar filas vacias (H25 en dias normales)
df_long = df_long.dropna(subset=["price"])

# Convertir a int
df_long["hour"] = df_long["hour"].astype(int)

# =========================
# 5. AJUSTAR HORAS
# =========================

# H1-H24 -> 0-23
# H25 -> tambien 23 (hora duplicada)
df_long["hour"] = df_long["hour"] - 1
# Si hay hora 24 (H25), la pasamos a 23 
df_long.loc[df_long["hour"] == 24, "hour"] = 23

# =========================
# 6. CREAR DATETIME
# =========================

df_long["datetime"] = pd.to_datetime(df_long["DATE"]) + \
                      pd.to_timedelta(df_long["hour"], unit="h")

# =========================
# 7. ELIMINAR DUPLICADOS (MEDIA)
# =========================

# Si hay dos horas 23 en cambio horario -> media
df_long = df_long.groupby(["datetime"], as_index=False)["price"].mean().round(2)

# =========================
# 8. AÑADIR HORA LIMPIA
# =========================

df_long["hour"] = df_long["datetime"].dt.hour

# =========================
# 9. LIMPIEZA FINAL
# =========================

df_long = df_long.sort_values("datetime").reset_index(drop=True)

print(df_long.head())

# =========================
# 10. COMPROBAR DUPLICADOS EN DATETIME
# =========================

duplicated_count = df_long["datetime"].duplicated().sum()
if duplicated_count > 0:
    print(f"ATENCIÓN: Hay {duplicated_count} filas duplicadas en datetime.")
    print("Filas duplicadas:")
    print(df_long[df_long["datetime"].duplicated(keep=False)].sort_values("datetime"))
else:
    print("No hay fechas duplicadas en datetime. Todo correcto.")


# =========================
# Guardarlo en un CSV
# =========================

df_long.to_csv("limpiezaPrecio1_Omie2020_2024.csv", sep=';', index=None)

              datetime  price  hour
0  2020-01-01 00:00:00  41.88     0
1  2020-01-01 01:00:00  38.60     1
2  2020-01-01 02:00:00  36.55     2
3  2020-01-01 03:00:00  32.32     3
4  2020-01-01 04:00:00  30.85     4
5  2020-01-01 05:00:00  30.14     5
6  2020-01-01 06:00:00  30.17     6
7  2020-01-01 07:00:00  30.00     7
8  2020-01-01 08:00:00  30.65     8
9  2020-01-01 09:00:00  30.65     9
10 2020-01-01 10:00:00  30.27    10
11 2020-01-01 11:00:00  30.34    11
12 2020-01-01 12:00:00  30.99    12
13 2020-01-01 13:00:00  30.04    13
14 2020-01-01 14:00:00  30.75    14
15 2020-01-01 15:00:00  32.11    15
16 2020-01-01 16:00:00  35.98    16
17 2020-01-01 17:00:00  40.40    17
18 2020-01-01 18:00:00  44.05    18
19 2020-01-01 19:00:00  46.16    19
20 2020-01-01 20:00:00  44.02    20
21 2020-01-01 21:00:00  45.60    21
22 2020-01-01 22:00:00  42.90    22
23 2020-01-01 23:00:00  37.55    23
No hay fechas duplicadas en datetime. Todo correcto.


C:\Users\Jaime_Sanchez\AppData\Local\Temp\ipykernel_9980\1360912298.py:65: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df_long = df_long.groupby(["datetime"], as_index=False)["price"].mean().round(2)


## Version 2
Intentar solucionar el problema del cambio de hora de primavera, falta una hora. Necesito interpolar (coger el valor medio de los vecinos)
Pra hacerlo, generar un rango de 23 horas, todos los dias tienen 23 horas, si uno le falta tendra un hueco, ese heuco se rellena con el precio que falta
NO FUNCIONA REVISAR 

In [ ]:
import pandas as pd

# 1. Cargar datos
df = pd.read_csv("OMIEPrecio2020_2024.csv", sep=";")

# 2. Filtrar precios
df = df[df["CONCEPT"] == "PRICE_SP"]

# 3. Pasar a formato largo
cols_horas = [c for c in df.columns if c.startswith("H")]

df = df.melt(
    id_vars=["DATE"],
    value_vars=cols_horas,
    var_name="hour",
    value_name="price"
)

# 4. Limpiar hora
df["hour"] = df["hour"].str.replace("H", "")
df["hour"] = pd.to_numeric(df["hour"], errors="coerce") # convertir a numero, si no se puede convertir se pone NaN

# eliminar filas vacias
df = df.dropna(subset=["price"])

df["hour"] = df["hour"].astype(int)

# convertir a rango 0-23
df["hour"] = df["hour"] - 1

# H25 -> convertir hora 24 a 23
df.loc[df["hour"] == 24, "hour"] = 23

# 5. Crear datetime
df["datetime"] = pd.to_datetime(df["DATE"]) + pd.to_timedelta(df["hour"], unit="h")

# 6. OTOÑO: eliminar duplicados con media
df = df.groupby("datetime", as_index=False)["price"].mean()

# 7. PRIMAVERA: rellenar horas faltantes
df = df.set_index("datetime")

# crear rango continuo de horas (desde la primera fecha hasta la ultima, con frecuencia de cada hora)
# esto asegura que si falta alguna hora, se crea una fila con NaN para esa hora, que luego se puede rellenar con interpolación
rango = pd.date_range(df.index.min(), df.index.max(), freq="h")

# reindexar, añade los huecos en el df para que luego se interpole su valor
df = df.reindex(rango)

# interpolar valores faltantes
df["price"] = df["price"].interpolate()

# volver a columna normal
df = df.reset_index().rename(columns={"index": "datetime"})

# 8. Obtener hora final
df["hour"] = df["datetime"].dt.hour

# 9. Ordenar
df = df.sort_values("datetime").reset_index(drop=True)

print(df.head())


# =========================
# Guardarlo en un CSV
# =========================

df_long.to_csv("limpiezaPrecio2_Omie2020_2024.csv", sep=';', index=None)

             datetime  price  hour
0 2020-01-01 00:00:00  41.88     0
1 2020-01-01 01:00:00  38.60     1
2 2020-01-01 02:00:00  36.55     2
3 2020-01-01 03:00:00  32.32     3
4 2020-01-01 04:00:00  30.85     4


## Version 3

In [1]:
import pandas as pd

# 1. Cargar datos
df = pd.read_csv("OMIEPrecio2020_2024.csv", sep=";")

# 2. Filtrar precios
df = df[df["CONCEPT"] == "PRICE_SP"]

# 3. Pasar a formato largo
cols_horas = [c for c in df.columns if c.startswith("H")]

df = df.melt(
    id_vars=["DATE"],
    value_vars=cols_horas,
    var_name="hour",
    value_name="price"
)

# 4. Limpiar hora
df["hour"] = df["hour"].str.replace("H", "")
df["hour"] = pd.to_numeric(df["hour"], errors="coerce")

# eliminar filas vacias
df = df.dropna(subset=["price"])

df["hour"] = df["hour"].astype(int)

# convertir a rango 0-23
df["hour"] = df["hour"] - 1

# H25 -> convertir hora 24 a 23 (cambio otoño), no importa que haya dos horas 23 en un dia luego las combinamos
df.loc[df["hour"] == 24, "hour"] = 23

# asegurar que price es numérico antes de hacer la media o interpolar
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# 5. Crear datetime
df["datetime"] = pd.to_datetime(df["DATE"]) + pd.to_timedelta(df["hour"], unit="h")

# 6. OTOÑO: eliminar duplicados con media de ambos valores repetidos
df = df.groupby("datetime", as_index=False)["price"].mean().round(2)

# 7. PRIMAVERA: rellenar horas faltantes
df = df.set_index("datetime")

# ordenar por si acaso (importante)
df = df.sort_index()

# crear rango continuo de horas
rango = pd.date_range(df.index.min(), df.index.max(), freq="h")

# reindexar -> aquí aparecen los huecos
df = df.reindex(rango)

# interpolar valores faltantes (lineal)
df["price"] = df["price"].interpolate(method="linear")

# volver a columna normal
df = df.reset_index().rename(columns={"index": "datetime"})

# 8. Obtener hora final
df["hour"] = df["datetime"].dt.hour

# 9. Ordenar
df = df.sort_values("datetime").reset_index(drop=True)

print(df.head())

# =========================
# Guardar CSV
# =========================
df.to_csv("limpiezaPrecio3_Omie2020_2024.csv", sep=';', index=False)

             datetime  price  hour
0 2020-01-01 00:00:00  41.88     0
1 2020-01-01 01:00:00  38.60     1
2 2020-01-01 02:00:00  36.55     2
3 2020-01-01 03:00:00  32.32     3
4 2020-01-01 04:00:00  30.85     4


C:\Users\Jaime_Sanchez\AppData\Local\Temp\ipykernel_6984\32633521.py:41: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df = df.groupby("datetime", as_index=False)["price"].mean().round(2)


## Explicacion paso a paso

2. Filtrar 
El dataset contiene el precio iberico incluido portugal, a nosotros solo nos intersa el precio español (SP)

3. Pasar las horas de un formato por columnas a filas (1 hora = 1 col --> 1 hora = 1 fila)
itera sobre las cols del df y selecciona aquellas cuyo nombre comienza con "H"

df = df.melt(...) -> Convierte el DataFrame al formato largo (tidy data) donde cada fila representa una unica fecha y hora.

Antes:
DATE        H1    H2    H3    H4    H5  ...  H24
2024-01-01  50    52    51    49    48  ...   55
2024-01-02  53    54    52    50    51  ...   56

Despues:
DATE        hour  price
2024-01-01  H1    50
2024-01-01  H2    52
2024-01-01  H3    51
...


6. OTOÑO: eliminar duplicados por cambio a horario invierno
df.groupby("datetime", as_index=False) -> Agrupa todas las filas que tienen el mismo valor en la columna "datetime"

as_index=False -> la columna "datetime" no se convierte en índice, sigue siendo una columna normal

["price"] -> Selecciona únicamente la columna "price" para aplicar la operación de agregación

.mean() -> Calcula la media aritmética de los precios dentro de cada grupo (cada datetime duplicado)

.round(2) -> Redondea a 2 decimales, porque en el dataset vienen así todos y sino se quedarian estos con mas deciamles que el resto
